In [ ]:
import numpy as np
from scipy.optimize import minimize
import matplotlib.pyplot as plt

# Given values
monthly_inventory_turnover = 3
cost_of_construction_per_sq_ft_private = 40
aisle_space = 0.5
number_of_years_amortization = 20
variability_provision = 0.7
variable_cost_per_kg_private_warehouses = 0.025
space_occupied_in_cu_ft_per_kg = 0.7
annual_fixed_cost = 6
stack_height_in_ft = 12
rental_charges_per_sq_ft_storage = 0.05
space_conversion_factor = 0.055555556
rental_handling_charges_per_sq_ft = 0.08

def calculate_allocated_space(private_space, space_requirement):
    """Calculate allocated space for the private warehouse."""
    allocated_space = np.minimum(private_space / space_requirement, 1)
    return allocated_space

def calculate_total_cost(sales_data, private_space):
    # Calculate space requirement
    space_requirement = np.array(sales_data) * space_conversion_factor
    
    # Calculate allocated space for private warehouse
    allocated_for_private_warehouse = calculate_allocated_space(private_space, space_requirement)
    
    # Calculate monthly fixed costs
    monthly_fixed_costs = ((private_space * allocated_for_private_warehouse * cost_of_construction_per_sq_ft_private) / number_of_years_amortization + (private_space * annual_fixed_cost)) / 12
    
    # Calculate monthly variable costs
    monthly_variable_costs = np.array(sales_data) * allocated_for_private_warehouse * variable_cost_per_kg_private_warehouses
    
    # Calculate monthly storage costs
    monthly_storage_costs = np.array(sales_data) * monthly_inventory_turnover * (1 - allocated_for_private_warehouse) * rental_charges_per_sq_ft_storage
    
    # Calculate monthly holding costs
    monthly_holding_costs = np.array(sales_data) * (1 - allocated_for_private_warehouse) * rental_handling_charges_per_sq_ft
    
    # Calculate total cost
    total_cost = np.sum(monthly_fixed_costs) + np.sum(monthly_variable_costs) + np.sum(monthly_storage_costs) + np.sum(monthly_holding_costs)
    
    return total_cost

def optimize_warehouse_sizing(sales_data):
    # Round off the maximum value of the sales data to the nearest 10,000
    total_area = round(max(sales_data), -4)
    
    # Define the objective function
    obj_func = lambda x: calculate_total_cost(sales_data, x[0] * 10000)
    
    # Initial guess for No
    x0 = [total_area / 10000]
    
    # Define bounds for No
    bounds = [(1, None)]
    
    # Perform optimization
    result = minimize(obj_func, x0=x0, bounds=bounds)
    
    # Extract optimal number of units (No)
    optimal_no = round(result.x[0])
    
    # Calculate private space requirement
    private_space_requirement = round(optimal_no * 10000)
    
    # Calculate total cost
    minimized_total_cost = result.fun
    
    return optimal_no, private_space_requirement, minimized_total_cost

def calculate_costs_for_all_nos(sales_data, max_no):
    """Calculate the total costs for all No values up to the maximum and return the list."""
    total_costs = []
    for No in range(1, max_no + 1):
        private_space = No * 10000
        total_cost = calculate_total_cost(sales_data, private_space)
        total_costs.append(total_cost)
    return total_costs

def plot_costs_vs_units(total_costs, optimal_no):
    """Plot the graph of total costs vs number of units with annotations for each point."""
    max_no = len(total_costs)
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, max_no + 1), total_costs, marker='o', label="Total Cost")
    plt.axvline(x=optimal_no, color='r', linestyle='--', label=f"Optimal No: {optimal_no:.0f}")
   
    # Annotate each point with its corresponding total cost
    for i, cost in enumerate(total_costs, start=1):
        plt.text(i, cost, f"{cost:.2f}", fontsize=9, ha='center', va='bottom')

    plt.xlabel("Number of Units (No)")
    plt.ylabel("Total Cost")
    plt.title("Total Costs vs Number of Units (No)")
    plt.xticks(range(1, max_no + 1))
    plt.legend()
    plt.grid(True)
    plt.show()

def print_costs_table(sales_data, private_space):
    """Print a table of costs for each month."""
    # Calculate space requirement
    space_requirement = np.array(sales_data) * space_conversion_factor
   
    # Calculate allocated space for private warehouse
    allocated_for_private_warehouse = calculate_allocated_space(private_space, space_requirement)
    rented = 1 - allocated_for_private_warehouse
   
    # Calculate monthly fixed costs
    monthly_fixed_costs = ((private_space * allocated_for_private_warehouse * cost_of_construction_per_sq_ft_private) / number_of_years_amortization + (private_space * annual_fixed_cost)) / 12
   
    # Calculate monthly variable costs
    monthly_variable_costs = np.array(sales_data) * allocated_for_private_warehouse * variable_cost_per_kg_private_warehouses
   
    # Calculate monthly storage costs
    monthly_storage_costs = np.array(sales_data) * monthly_inventory_turnover * rented * rental_charges_per_sq_ft_storage
   
    # Calculate monthly holding costs
    monthly_holding_costs = np.array(sales_data) * rented * rental_handling_charges_per_sq_ft
   
    # Print all costs in a tabular format
    print("Month\tSpace Requirement\tAllocated for Private Warehouse\tRented\tFixed Cost\tVariable Cost\tStorage Cost\tHolding Cost\tTotal Cost")
    for i in range(len(sales_data)):
        total_monthly_cost = monthly_fixed_costs[i] + monthly_variable_costs[i] + monthly_storage_costs[i] + monthly_holding_costs[i]
        print(f"{i+1}\t{space_requirement[i]:.2f}\t\t{allocated_for_private_warehouse[i]:.2f}\t\t{rented[i]:.2f}\t{monthly_fixed_costs[i]:.2f}\t{monthly_variable_costs[i]:.2f}\t{monthly_storage_costs[i]:.2f}\t{monthly_holding_costs[i]:.2f}\t{total_monthly_cost:.2f}")

if __name__ == "__main__":
    # Input sales data month by month
    sales_data = []
    for month in range(1, 13):
        while True:
            try:
                sales = float(input(f"Enter sales data for month {month}: "))
                if sales < 0:
                    raise ValueError("Sales data cannot be negative.")
                sales_data.append(sales)
                break
            except ValueError as e:
                print(e)
    

    # Optimize warehouse sizing
    optimal_no, private_space, minimized_total_cost = optimize_warehouse_sizing(sales_data)
    
    # Print results
    print("Optimal Number of Units (No):", optimal_no)
    print("Private Space Requirement:", private_space)
    print("Minimized Total Cost:", minimized_total_cost)
    
    # Print costs table
    print("\nCosts for each month:")
    print_costs_table(sales_data, private_space)
    
    # Plot costs vs units
    max_no = 7
    total_costs = calculate_costs_for_all_nos(sales_data, max_no)
    plot_costs_vs_units(total_costs, optimal_no)